# TaaSim Week 5: Batch ETL & Spark Analytics Overview

This notebook provides a detailed walkthrough of the tasks completed during **Week 5**, focusing on building a robust, scalable data pipeline using Apache Spark.

---

## 🎯 1. Engineering Tasks & Objectives

Based on the **ID2-ABD-Capstone project-2026.pdf**, the following objectives were met:

### A. Spark ETL Job (Casablanca Data)
- **Source**: Raw trip data from `data/raw/casablanca_real_roads_final.csv`.
- **Processing**:
    - Parsed the `CASA_POLYLINE` JSON string to extract spatial coordinates.
    - Computed **H3 Geospatial Indices** (Resolution 8) for every pickup and dropoff point.
    - Deduplicated and cleaned the dataset.
- **Sink**: Saved as optimized Parquet files in MinIO `curated/` bucket.

### B. Spark SQL Analytics (Weekly KPIs)
We developed transformations to extract actionable insights:
- **Trips per Zone**: Aggregating demand density by H3 cell.
- **Peak Demand Hours**: Temporal analysis of trip timestamps.
- **Coverage Gap**: Identifying zones with high demand but low vehicle supply.

### C. Persistence & Visualization
- **Cassandra**: Persisted the KPI aggregates into the `demand_zones` table for fast API access.
- **Grafana**: (Infrastructure setup) Ready to visualize demand heatmaps and peak hour charts.

## 🛠 2. Implementation Details

The following scripts were implemented to fulfill these requirements:

### `scripts/jobs/casablanca_batch_etl.py`
This script handles the raw-to-curated pipeline. Key logic includes:
```python
# Parsing Polyline and Computing H3
df_transformed = df_raw.withColumn("coords", parse_polyline_udf(col("CASA_POLYLINE"))) \
    .withColumn("pickup_h3", h3_udf(col("coords")[0][0], col("coords")[0][1]))
```

### `scripts/jobs/mobility_kpi_analyzer.py`
This script aggregates the curated data. Key logic includes:
```python
# Grouping by Zone and Hour
df_zone_demand = df_curated.groupBy("pickup_h3").count()
df_peak_hours = df_curated.withColumn("hour", hour(from_unixtime(col("TIMESTAMP")))) \
    .groupBy("hour").count()
```

## 📈 3. Visualizing Results

You can use the existing `notebooks/Week5_Analytics_Exploration.ipynb` to interactively visualize these patterns on a Casablanca map using `folium`.

---
**End of Week 5 Summary**